## Task 3

In [ ]:
import numpy as np


def removing_price_outliers(input_price_list: list) -> np.ndarray:
    """
    Converts a list of price values to a 1D NumPy array and replaces all hardware glitch outliers
    with the median of the valid prices.

    Args:
        input_price_list: A list of price values.
    Returns:
        cleaned_price_array: A 1D NumPy array with valid prices.
    """

    # Convert the input price data list to a 1D NumPy float array
    input_price_array = np.array(input_price_list)

    # Calculate the mean and standard deviation of the array with prices
    mean = np.mean(input_price_array)
    sigma = np.std(input_price_array)

    # Find all valid prices to later calculate the median of the valid prices
    valid_prices = []
    for price in input_price_array:
        if abs(price - mean) <= 3 * sigma:
            valid_prices.append(price)

    # Calculate the median of the valid prices
    median = np.median(valid_prices)

    # Replace all hardware glitch outliers with the median of the valid prices
    cleaned_price_array = np.where(abs(input_price_array - mean) > 3 * sigma, median, input_price_array)

    return cleaned_price_array


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculates the refund ratio"""

    # We have 10**(-6) in the denominator to prevent zero-devision errors
    refund_ratio = refund_count / (sales_count + 10**(-6))

    return refund_ratio



## Task 4

In [50]:
import pandas as pd


def list_to_dataframe(transaction_data: list) -> pd.DataFrame:
    """Converts a list with dictionaries to a pandas DataFrame and.
    Converts the timestamp to datetime and sets it as the index of the DataFrame."""

    # Convert the list to a DataFrame
    df_transaction_data = pd.DataFrame(transaction_data)

    # Convert the timestamp column to datetime
    df_transaction_data['timestamp'] = pd.to_datetime(df_transaction_data['timestamp'])

    # Set the datetime as index
    df_transaction_data = df_transaction_data.set_index('timestamp')

    # Sort the DataFrame chronologically by the timestamp
    df_transaction_data = df_transaction_data.sort_index()

    return df_transaction_data

list_2 = [
    {'timestamp': '2026-10-15 07:01:54', 'register_ID': 'POS_02', 'item': 'Cappuccino', 'price': 3.89, 'currency': 'EUR', 'status': 'CANCELLED'},
    {'timestamp': '2026-10-15 07:04:25', 'register_ID': 'POS_02', 'item': 'Muffin', 'price': 3.09, 'currency': 'USD', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 07:46:33', 'register_ID': 'POS_02', 'item': 'Muffin', 'price': 3.53, 'currency': 'EUR', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 08:21:30', 'register_ID': 'POS_02', 'item': 'Cappuccino', 'price': 3.72, 'currency': 'NOK', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 08:26:10', 'register_ID': 'POS_02', 'item': 'Muffin', 'price': 3.84, 'currency': 'EUR', 'status': 'CANCELLED'},
    {'timestamp': '2026-10-15 08:28:09', 'register_ID': 'POS_02', 'item': 'Cappuccino', 'price': 3.93, 'currency': 'NOK', 'status': 'REFUND'}
]

print(list_to_dataframe(list_2))


def group_by_item(df_data: pd.DataFrame) -> pd.DataFrame:
    """Groups transaction data by item and resamples transactions to one-hour windows.
    Calculate the total prices per item per hour."""

    # Group data by item
    grouped_by_item = df_data.groupby('item')

    # Resample transactions in one-hour window and calculte the sum of the prices
    hourly_sum = grouped_by_item.resample('1h').sum()

    return hourly_sum


print(group_by_item(list_to_dataframe(list_2)))


def refund_ratio_per_item(df_data: pd.DataFrame) -> None:
    """"""

    sales_count = []
    refund_count = []

    for item in df_data.groupby('item'):
        sales = 0
        refund = 0

        for transaction in item:
            sales += 1

            if df_data.groupby('item')['status'] == 'REFUND':
                refund += 1

        sales_count.append(sales)
        refund_count.append(refund)

    sales_count = np.array(sales_count)
    refund_count = np.array(refund_count)

    refund_ratio_array = compute_refund_ratio(sales_count, refund_count)

    return refund_ratio_array


print(refund_ratio_per_item(list_to_dataframe(list_2)))


                    register_ID        item  price currency     status
timestamp                                                             
2026-10-15 07:01:54      POS_02  Cappuccino   3.89      EUR  CANCELLED
2026-10-15 07:04:25      POS_02      Muffin   3.09      USD    SUCCESS
2026-10-15 07:46:33      POS_02      Muffin   3.53      EUR    SUCCESS
2026-10-15 08:21:30      POS_02  Cappuccino   3.72      NOK    SUCCESS
2026-10-15 08:26:10      POS_02      Muffin   3.84      EUR  CANCELLED
2026-10-15 08:28:09      POS_02  Cappuccino   3.93      NOK     REFUND
                                 register_ID  price currency          status
item       timestamp                                                        
Cappuccino 2026-10-15 07:00:00        POS_02   3.89      EUR       CANCELLED
           2026-10-15 08:00:00  POS_02POS_02   7.65   NOKNOK   SUCCESSREFUND
Muffin     2026-10-15 07:00:00  POS_02POS_02   6.62   USDEUR  SUCCESSSUCCESS
           2026-10-15 08:00:00        POS_02   